# 02 — ToN-IoT Feature Selection

**Goal:** Review the 10 candidate network-flow features without deleting any records.

**Input:** Original ToN-IoT CSV and Notebook 01 audit.

**Output:** Reduced Core10 CSV.GZ, feature-quality report and one graph.

**Labels:** `label` and `type` are retained for evaluation, **not** used as model inputs.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
RAW = ROOT / 'data/raw/ton_iot/ton_iot_train_test_network.csv'
AUDIT1 = ROOT / 'reports/audits/ton_iot_v3_dataset_analysis.json'
OUT = ROOT / 'data/processed/ton_iot/ton_iot_core10_selected_v3.csv.gz'
REPORT = ROOT / 'reports/audits/ton_iot_v3_feature_quality.json'
FIGURE = ROOT / 'reports/figures/ton_iot_v3_feature_missingness.png'

EXPECTED_ROWS = 211043
CATEGORICAL = ['proto', 'service', 'conn_state']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown'}

for path in (RAW, AUDIT1):
    if not path.is_file():
        raise FileNotFoundError(f'Missing Notebook 01 input: {path}')
audit = json.loads(AUDIT1.read_text(encoding='utf-8'))
FEATURES = audit['candidate_flow_features']
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
if len(FEATURES) != 10 or len(NUMERIC) != 7 or not set(CATEGORICAL).issubset(FEATURES):
    raise ValueError('Unexpected 10-feature schema; review Notebook 01.')
if audit['rows'] != EXPECTED_ROWS:
    raise ValueError(f'Notebook 01 row count changed: {audit["rows"]}')
FIELDS = ['csv_row_index_0based'] + FEATURES + ['label', 'type']
print('Features:', len(FEATURES), '| Source records:', EXPECTED_ROWS)
print('All rows will be retained. Source CSV remains unchanged.')

## 1. Read all records

Read the ten selected columns and two labels. No filtering or sampling.

In [ ]:
sha = hashlib.sha256()
with RAW.open('rb') as file:
    for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
        sha.update(block)
source_sha = sha.hexdigest()
if source_sha != audit['source_sha256']:
    raise ValueError('Source CSV differs from Notebook 01. Do not continue.')

source_columns = pd.read_csv(RAW, nrows=0).columns.tolist()
if any(c not in source_columns for c in FEATURES + ['label', 'type']):
    raise ValueError('Required columns are absent from the source CSV.')

frame = pd.read_csv(RAW, usecols=FEATURES + ['label', 'type'],
                    dtype=str, keep_default_na=False, low_memory=False)
frame = frame[FEATURES + ['label', 'type']].copy()
if len(frame) != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} rows; found {len(frame):,}.')
frame.insert(0, 'csv_row_index_0based', np.arange(len(frame), dtype=np.int64))
frame['label'] = frame['label'].str.strip()
frame['type'] = frame['type'].str.strip()
if set(frame['label']) != {'0', '1'}:
    raise ValueError('Expected binary labels 0=Normal, 1=Attack.')
observed_labels = frame['label'].value_counts().to_dict()
if observed_labels != {str(k): int(v) for k,v in audit['label_counts'].items()}:
    raise ValueError('Label totals differ from Notebook 01.')
label_mismatch = ((frame['label'] == '0') != frame['type'].str.casefold().eq('normal')).sum()
if label_mismatch:
    raise ValueError(f'Binary/type label mismatches: {label_mismatch}')
print(f'Loaded: {len(frame):,}/{EXPECTED_ROWS:,} records (100%)')
print('Labels:', observed_labels)
print('Export columns:', len(FIELDS), '(10 features + 2 labels + row ID)')

## 2. Feature-quality checks

Count missing values, invalid numbers and differences between Normal and Attack.

In [ ]:
rows = []
canonical = pd.DataFrame(index=frame.index)
for feature in FEATURES:
    raw = frame[feature].astype('string').str.strip()
    missing = raw.str.casefold().isin(MISSING).fillna(True)
    if feature in NUMERIC:
        values = pd.to_numeric(raw.mask(missing), errors='coerce')
        invalid = (~missing) & values.isna()
        # Normalize equivalent numeric text before repeated-feature checks.
        canonical[feature] = values.astype('string').fillna('__MISSING__')
    else:
        invalid = pd.Series(False, index=frame.index)
        canonical[feature] = raw.str.casefold().mask(missing, '__MISSING__').fillna('__MISSING__')

    normal = frame['label'].eq('0')
    attack = frame['label'].eq('1')
    rows.append({
        'feature': feature,
        'type': 'categorical' if feature in CATEGORICAL else 'numeric',
        'missing_count': int(missing.sum()),
        'missing_pct': round(float(missing.mean() * 100), 2),
        'invalid_numeric_count': int(invalid.sum()),
        'normal_missing_pct': round(float(missing[normal].mean() * 100), 2),
        'attack_missing_pct': round(float(missing[attack].mean() * 100), 2)
    })
quality = pd.DataFrame(rows).sort_values('missing_pct', ascending=False)
display(quality)
print('The Service placeholder needs review; do not automatically remove the feature.')

## 3. Repeated feature patterns

Check whether identical normalized features have different labels. Keep all records.

In [ ]:
# Pandas 64-bit fingerprints provide an efficient duplicate screen.
# A fingerprint match is not proof of session/capture independence.
signatures = pd.util.hash_pandas_object(canonical, index=False).to_numpy(dtype=np.uint64)
groups = pd.DataFrame({'signature': signatures, 'label': frame['label'].to_numpy()})
stats = groups.groupby('signature', sort=False).agg(
    rows=('label', 'size'), label_count=('label', 'nunique')
)
repeated = stats['rows'] > 1
conflicting = stats['label_count'] > 1
result = {
    'total_rows': len(frame),
    'distinct_normalized_feature_patterns': int(len(stats)),
    'repeated_feature_groups': int(repeated.sum()),
    'rows_in_repeated_groups': int(stats.loc[repeated, 'rows'].sum()),
    'conflicting_binary_label_groups': int(conflicting.sum()),
    'rows_in_conflicting_groups': int(stats.loc[conflicting, 'rows'].sum()),
}
for key, value in result.items():
    print(f'{key}: {value:,}')
print('These are feature-pattern diagnostics, not a reason to drop repeated records.')

## 4. Save the reduced dataset

Keep all 211,043 records and their original labels. Split the data in Notebook 03.

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)
FIGURE.parent.mkdir(parents=True, exist_ok=True)
temporary = Path(str(OUT) + '.tmp')
if temporary.exists():
    raise FileExistsError(f'Previous incomplete export found: {temporary}')

frame[FIELDS].to_csv(temporary, index=False, compression='gzip')
# Verify data written to disk before replacing any previous output.
read_rows = 0
expected_next_id = 0
for chunk in pd.read_csv(temporary, dtype=str, keep_default_na=False,
                         compression='gzip', chunksize=30000):
    if chunk.columns.tolist() != FIELDS:
        raise ValueError('Exported column order changed.')
    indexes = pd.to_numeric(chunk['csv_row_index_0based'], errors='raise').to_numpy()
    if not np.array_equal(indexes, np.arange(expected_next_id, expected_next_id + len(chunk))):
        raise ValueError('Exported row IDs are missing or out of order.')
    expected_next_id += len(chunk)
    read_rows += len(chunk)
if read_rows != EXPECTED_ROWS:
    raise ValueError(f'Only {read_rows:,} rows were exported.')
temporary.replace(OUT)
print(f'Saved: {OUT}')
print(f'Rows: {read_rows:,} (100%) | Columns: {len(FIELDS)}')

## 5. Graph and report

Save one feature-quality graph and a short experiment report.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
chart = quality.sort_values('missing_pct').copy()
ax.barh(chart['feature'], chart['missing_pct'])
ax.set_xlabel('Missing or placeholder values (%)')
ax.set_title('ToN-IoT — Core10 feature missingness')
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURE, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

report = {
    'dataset': 'ToN-IoT network CSV',
    'source_file': RAW.name,
    'source_sha256_from_notebook_01': source_sha,
    'rows': len(frame),
    'selected_features': FEATURES,
    'categorical_features': CATEGORICAL,
    'numeric_features': NUMERIC,
    'target': 'label',
    'additional_label': 'type',
    'exported_columns': FIELDS,
    'export_file': OUT.name,
    'exported_rows': read_rows,
    'feature_quality': quality.to_dict(orient='records'),
    'feature_pattern_diagnostics': result,
    'limitations': [
        'Feature names alone do not verify equivalence with X-IIoTID.',
        'Session/capture independence is not verified.',
        'Service placeholders may be informative; validate missingness-only baselines.'
    ],
    'test_scored': False,
    'split_performed': False
}
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved:', REPORT)
print('Saved:', FIGURE)
print('Next: 03_data_preparation.ipynb — create train/validation/test splits with Excel copies.')